# Boltz-2: Binding Affinity **From a Provided Structure**

![Boltz2](https://img.shields.io/badge/Model-Boltz2-purple) ![Colab](https://img.shields.io/badge/Platform-Colab%20%7C%20Linux-lightgrey?logo=googlecolab) ![License](https://img.shields.io/badge/License-MIT-orange)

Feed **any structure (CIF, including multimers)** and predict the **binding affinity**
of a ligand against it — with a one-click affinity toggle.

### What this does
Boltz-2 always runs its diffusion structure module; it doesn't score a pre-built
complex directly. So we feed *your* structure through Boltz-2's **`templates:`** block:

1. Protein chains + sequences are **auto-extracted from your CIF**.
2. Your structure is used as a **template** for those chains (optionally *forced*,
   restraining the backbone to your coordinates).
3. The ligand is given as **SMILES/CCD** and flagged as the affinity **`binder`**.
4. Optionally a **pocket constraint** pins the ligand to the real site.

### Affinity caveats (read once)
- Affinity = **one small-molecule ligand only** (no protein–protein, no multi-ligand).
- The affinity head does **not** explicitly model cofactors/ions/water/multimeric
  partners → for multimers, read the number as *"ligand vs. the templated pocket"*
  and validate against MM/GBSA / ABFE / MST.
- Use **CIF** (PDB templates are buggy upstream). Ligands with **≥128 atoms** are rejected for affinity.

**Runtime → Change runtime type → GPU** before you start.


In [ ]:
#@title 1. Install Boltz-2 + dependencies, check GPU
import subprocess, sys, torch
print("GPU available:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
if not torch.cuda.is_available():
    print("WARNING: no GPU. Set Runtime > Change runtime type > GPU (T4 or better).")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "boltz", "gemmi", "pyyaml", "py3Dmol"], check=True)
print("Boltz-2 + deps installed.")


In [ ]:
#@title 2. Helper functions (self-contained — same logic as the repo library)
from dataclasses import dataclass, field
from typing import Optional, Union
import gemmi, yaml, glob, json, os

@dataclass
class ProteinChain:
    id: str; sequence: str; msa: Optional[str] = None
@dataclass
class Ligand:
    id: str; smiles: Optional[str] = None; ccd: Optional[str] = None
@dataclass
class TemplateSpec:
    cif: str; chain_id=None; template_id=None; force: bool=False; threshold: Optional[float]=None
@dataclass
class PocketConstraint:
    binder: str; contacts: list; max_distance: float = 6.0

def parse_structure(path):
    st = gemmi.read_structure(path); st.setup_entities(); model = st[0]
    chains, ligands = [], []
    for chain in model:
        poly = chain.get_polymer()
        if len(poly):
            pt = poly.check_polymer_type()
            kind = "protein" if pt in (gemmi.PolymerType.PeptideL, gemmi.PolymerType.PeptideD) else (
                   "nucleic" if pt in (gemmi.PolymerType.Dna, gemmi.PolymerType.Rna, gemmi.PolymerType.DnaRnaHybrid) else "other")
            seq = "".join(c for c in poly.make_one_letter_sequence() if c.isalpha())
            chains.append({"chain_id": chain.name, "kind": kind, "sequence": seq, "n": len(poly)})
        for res in chain.get_ligands():
            ligands.append({"chain_id": chain.name, "resname": res.name, "n_atoms": len(res)})
    return {"path": path, "chains": chains, "ligands": ligands}

def build_yaml_dict(proteins, ligands, binder_id=None, templates=None, pocket=None, version=1):
    seqs = []
    for p in proteins:
        e = {"id": p.id, "sequence": p.sequence}
        if p.msa: e["msa"] = p.msa
        seqs.append({"protein": e})
    for lig in ligands:
        e = {"id": lig.id}
        e["smiles" if lig.smiles else "ccd"] = lig.smiles or lig.ccd
        seqs.append({"ligand": e})
    out = {"version": version, "sequences": seqs}
    if templates:
        tl = []
        for t in templates:
            td = {"cif": t.cif}
            if t.chain_id is not None: td["chain_id"] = t.chain_id
            if t.template_id is not None: td["template_id"] = t.template_id
            if t.force:
                td["force"] = True
                assert t.threshold is not None, "force=True needs threshold"
                td["threshold"] = t.threshold
            tl.append(td)
        out["templates"] = tl
    if pocket is not None:
        out["constraints"] = [{"pocket": {"binder": pocket.binder,
            "contacts": [list(c) for c in pocket.contacts], "max_distance": pocket.max_distance}}]
    if binder_id is not None:
        out["properties"] = [{"affinity": {"binder": binder_id}}]
    return out

def _pred_dir(out_dir):
    c = [d for d in glob.glob(os.path.join(out_dir,"**","predictions","*"), recursive=True) if os.path.isdir(d)]
    if not c: raise FileNotFoundError("no predictions dir")
    return c[0]

def load_affinity(out_dir):
    m = glob.glob(os.path.join(_pred_dir(out_dir), "affinity_*.json"))
    if not m: raise FileNotFoundError("no affinity_*.json - did you enable affinity?")
    d = json.load(open(m[0]))
    v = float(d["affinity_pred_value"])
    return {"pred_value": v, "probability_binary": float(d["affinity_probability_binary"]),
            "pred_value1": d.get("affinity_pred_value1"), "pred_value2": d.get("affinity_pred_value2"),
            "ic50_uM": 10.0**v, "pic50": 6.0-v, "dg": (6.0-v)*1.364}

def top_structure(out_dir):
    pd = _pred_dir(out_dir)
    for pat in ("*_model_0.cif","*_model_0.pdb","*.cif","*.pdb"):
        m = sorted(glob.glob(os.path.join(pd, pat)))
        if m: return m[0]
    return None

print("Helpers ready.")


In [ ]:
#@title 3. Upload your structure (CIF recommended)
from google.colab import files
import os
os.makedirs("/content/work", exist_ok=True)
print("Select a .cif (or .pdb) file of your complex/multimer...")
up = files.upload()
STRUCTURE_PATH = "/content/work/" + list(up.keys())[0]
with open(STRUCTURE_PATH, "wb") as f:
    f.write(up[list(up.keys())[0]])

PARSED = parse_structure(STRUCTURE_PATH)
print(f"\nParsed: {STRUCTURE_PATH}\nChains:")
for c in PARSED["chains"]:
    prev = (c["sequence"][:50] + "...") if len(c["sequence"]) > 50 else c["sequence"]
    print(f"  [{c['chain_id']}] {c['kind']:8s} {c['n']:>4d} res  {prev}")
print("Ligand/non-polymer residues:",
      ", ".join(f"{l['chain_id']}:{l['resname']}({l['n_atoms']})" for l in PARSED["ligands"]) or "none")
PROTEIN_CHAINS = [c for c in PARSED["chains"] if c["kind"] == "protein"]
print(f"\n-> {len(PROTEIN_CHAINS)} protein chain(s) will be templated:",
      [c["chain_id"] for c in PROTEIN_CHAINS])


In [ ]:
#@title 4. Configure ligand + affinity options { run: "auto" }
#@markdown **Ligand** (the molecule whose affinity you want):
ligand_input_type = "smiles"  #@param ["smiles", "ccd"]
ligand_value = "N[C@@H](Cc1ccc(O)cc1)C(=O)O"  #@param {type:"string"}
ligand_id    = "L"  #@param {type:"string"}

#@markdown **Affinity** (the one-click toggle):
predict_affinity = True  #@param {type:"boolean"}

#@markdown **Template behaviour** — use your uploaded structure to fix the protein pose:
use_structure_as_template = True  #@param {type:"boolean"}
force_backbone_to_template = True  #@param {type:"boolean"}
force_threshold_A = 5.0  #@param {type:"number"}

#@markdown **Optional pocket constraint** (recommended for reliable affinity).
#@markdown Format: `Chain:resi` comma-separated, e.g. `A:34, A:56, B:18`. Leave blank to skip.
pocket_contacts = ""  #@param {type:"string"}

#@markdown **MSA**: let Boltz fetch MSAs automatically (needed unless you provide your own).
use_msa_server = True  #@param {type:"boolean"}

print("Ligand:", ligand_id, "=", ligand_value, f"({ligand_input_type})")
print("Affinity:", predict_affinity, "| Template:", use_structure_as_template,
      "| Force:", force_backbone_to_template)
print("Pocket contacts:", pocket_contacts or "(none)")


In [ ]:
#@title 5. Build the Boltz-2 input YAML
proteins = [ProteinChain(c["chain_id"], c["sequence"]) for c in PROTEIN_CHAINS]
used = {p.id for p in proteins}
lig_id = ligand_id if ligand_id not in used else next(x for x in "LMNOPQRSTUVWXYZ" if x not in used)
ligand = Ligand(lig_id, smiles=ligand_value if ligand_input_type=="smiles" else None,
                        ccd=ligand_value if ligand_input_type=="ccd" else None)

templates = None
if use_structure_as_template:
    templates = [TemplateSpec(cif=STRUCTURE_PATH, chain_id=[p.id for p in proteins],
                              force=force_backbone_to_template,
                              threshold=force_threshold_A if force_backbone_to_template else None)]

pocket = None
if pocket_contacts.strip():
    contacts = []
    for tok in pocket_contacts.split(","):
        ch, ri = tok.strip().split(":")
        contacts.append([ch.strip(), int(ri)])
    pocket = PocketConstraint(binder=lig_id, contacts=contacts)

data = build_yaml_dict(proteins, [ligand],
                       binder_id=lig_id if predict_affinity else None,
                       templates=templates, pocket=pocket)

YAML_PATH = "/content/work/input.yaml"
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)
print(open(YAML_PATH).read())


In [ ]:
#@title 6. Run Boltz-2
import subprocess, os
OUT_DIR = "/content/work/results"
os.makedirs(OUT_DIR, exist_ok=True)
cmd = ["boltz", "predict", YAML_PATH, "--out_dir", OUT_DIR,
       "--accelerator", "gpu", "--devices", "1",
       "--diffusion_samples", "1", "--output_format", "mmcif"]
if use_msa_server:
    cmd.append("--use_msa_server")
print("Running:", " ".join(cmd), "\n(first run downloads model weights — be patient)\n")
rc = subprocess.run(cmd).returncode
print("\nDone." if rc == 0 else f"\nFAILED (exit {rc}) — check the log above.")


In [ ]:
#@title 7. Affinity dashboard
import matplotlib.pyplot as plt
if predict_affinity:
    a = load_affinity(OUT_DIR)
    print("=== Boltz-2 affinity (binder:", lig_id, ") ===")
    print(f"affinity_pred_value  : {a['pred_value']:.3f}   (log10 IC50 in uM; lower = tighter)")
    print(f"  IC50               : {a['ic50_uM']:.3g} uM")
    print(f"  pIC50              : {a['pic50']:.2f}")
    print(f"  dG (approx)        : {a['dg']:.2f} kcal/mol  (non-standard form)")
    print(f"affinity_probability : {a['probability_binary']:.3f}   (P(binder), 0..1)")
    if a['pred_value1'] is not None:
        print(f"  ensemble values    : {a['pred_value1']:.3f} / {a['pred_value2']:.3f}")
    print("\nUse probability for hit-vs-decoy; pred_value for SAR / lead-opt.")

    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].bar(["P(binder)"], [a["probability_binary"]], color="#6c5ce7")
    ax[0].set_ylim(0, 1); ax[0].set_title("Binder probability")
    ax[1].bar(["pIC50"], [a["pic50"]], color="#00b894")
    ax[1].set_title("pIC50 (higher = tighter)")
    plt.tight_layout(); plt.show()
else:
    print("Affinity was disabled in step 4 (predict_affinity = False).")


In [ ]:
#@title 8. View the predicted 3D structure
import py3Dmol
s = top_structure(OUT_DIR)
print("Structure:", s)
view = py3Dmol.view(width=720, height=520)
view.addModel(open(s).read(), "cif")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addStyle({"hetflag": True}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo(); view.show()


In [ ]:
#@title 9. Download all results (.zip)
import shutil
from google.colab import files
zip_path = "/content/boltz2_affinity_results"
shutil.make_archive(zip_path, "zip", OUT_DIR)
files.download(zip_path + ".zip")
print("Downloaded boltz2_affinity_results.zip")
